# 🔬 Notebook 3: ChatGPT — Conversation State & Reliability


## 🛠️ Setup

```bash
cd 06-system-designs/chatgpt
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

Stdlib only, seeded, deterministic. Run cells top-to-bottom.


## 💡 Everything around the model

Notebook 1 sized the fleet, notebook 2 built the scheduler. This notebook is about the parts
of the system that would exist even if the GPU were free — and which are, in interview terms,
where the design is actually won or lost.

1. **Conversation storage** and the context-window problem: the history grows without bound,
   the model's input does not.
2. **Metering and quotas** — in tokens, because requests are not the unit of cost.
3. **Idempotency and retries for a partially-delivered stream.** This is the genuinely hard
   one and it has no clean answer.
4. **Caching** — exact, prefix, and semantic, and when semantic caching lies to your users.
5. **Safety** as a pipeline stage, and what it costs in latency.


## 🗄️ Conversation storage

The data model is boring, and that's the point — it should be. What is interesting is that
the read pattern for *serving a turn* is different from the read pattern for *the sidebar*,
and conflating them is how you end up with a slow product.

```sql
-- The hot path. One partition per conversation; a turn is a single-partition range scan.
CREATE TABLE messages (
    conversation_id  uuid,        -- partition key
    created_at       timestamp,   -- clustering key, ascending
    message_id       uuid,        -- UUIDv7 so it also sorts by time
    role             text,        -- 'system' | 'user' | 'assistant' | 'tool'
    content          text,
    token_count      int,         -- denormalised: needed to build the prompt, never recount
    status           text,        -- 'complete' | 'streaming' | 'failed' | 'moderated'
    model            text,
    PRIMARY KEY ((conversation_id), created_at, message_id)
);

-- The sidebar. A separate table, because "list my conversations" must NOT scan messages.
CREATE TABLE conversations_by_user (
    user_id          uuid,        -- partition key
    updated_at       timestamp,   -- clustering key, DESCENDING
    conversation_id  uuid,
    title            text,        -- generated once from the first turn, then frozen
    PRIMARY KEY ((user_id), updated_at, conversation_id)
) WITH CLUSTERING ORDER BY (updated_at DESC);
```

Three decisions worth defending:

- **`token_count` is denormalised onto every message.** Building a prompt means summing token
  counts until you hit the budget. Re-tokenising the history on every turn would add real CPU
  to the hot path for information you already computed once.
- **`status` exists because generation can fail halfway.** A message is written *before* it is
  complete (see the streaming section), so "half an assistant reply, marked incomplete" must
  be a representable state. Systems that only persist completed messages lose the answer
  whenever a replica dies mid-generation.
- **Two tables, denormalised, no join.** `conversations_by_user` is updated on write. This is
  a wide-column data model, and the reason is that the entity boundary (`conversation_id`) is
  also the only access boundary — there is no query that spans conversations on the hot path.

⚖️ **What this costs:** dual writes. Appending a message updates two tables and they can
diverge if the second write fails. Fix it the boring way — the sidebar row is derivable, so
repair it lazily on read or from a periodic reconciliation job. Do **not** reach for a
distributed transaction to keep a sidebar timestamp accurate.


## 📏 The context-window problem

A 40-turn conversation is tens of thousands of tokens. The model has a fixed window, and every
token you send is paid for twice — once in prefill compute (notebook 1: prompt work grows
*quadratically* with turn count) and once in KV-cache memory (notebook 2: the real capacity
limit).

So you must choose what to leave out. Three strategies, and they lose **different things**:

| Strategy | Keeps | Loses | Extra cost |
|---|---|---|---|
| **Truncate** to the last N turns | perfect recent detail | everything older, silently | none |
| **Summarise** the old prefix | global gist of the whole conversation | detail and numeric precision | an extra LLM call |
| **Retrieve** relevant old turns | targeted old detail, verbatim | anything the query doesn't match | embedding index + lookup |

Let's measure that, rather than assert it.


In [ ]:
import random

# ==== ASSUMPTIONS: context policy ====
CONTEXT_BUDGET_TOKENS  = 4_000   # what we're willing to send as prompt   [range: 1k - 200k]
SUMMARY_TOKENS         = 300     # size of the rolling summary
RECENT_VERBATIM_TURNS  = 4       # turns always kept word-for-word
N_TURNS                = 40

# ASSUMPTIONS about summarisation QUALITY. There is no LLM here, so we model what a
# summariser does: it keeps some facts and drops others, and it rounds off numbers.
# These two numbers are the honest weak point of this simulation -- state them, don't hide
# them, and note that in production you would measure them with an eval set.
SUMMARY_FACT_RETENTION = 0.60    # P(a given fact survives summarisation)  [range: 0.3 - 0.9]
SUMMARY_PRECISION_LOSS = 0.50    # P(a surviving NUMBER loses its exact value)

FILLER = ['thanks that helps', 'can you expand on that', 'ok noted',
          'what about the other option', 'makes sense', 'go on',
          'let me think about that', 'and after that', 'right',
          'could you rephrase that', 'interesting', 'one more thing']

# (turn, message, fact key, is the value numeric?)
FACTS = [
    (2,  'just so you know I am allergic to peanuts',   'allergy',     False),
    (5,  'our project deadline is March 14',            'deadline',    False),
    (9,  'the total budget is 4200 euros',              'budget',      True),
    (13, 'my co-founder is named Dana',                 'cofounder',   False),
    (18, 'we deploy everything to eu-west-1',           'region',      False),
    (24, 'our error budget is 0.1 percent per month',   'errorbudget', True),
    (31, 'the customer for this is Acme Corp',          'customer',    False),
    (37, 'please always answer me in bullet points',    'style',       False),
]

# Questions asked at turn 41. Note that some share words with the message that answers
# them and some deliberately do not -- that vocabulary gap is what breaks retrieval.
PROBES = [
    ('what food should I avoid',      'allergy'),
    ('when is the deadline',          'deadline'),
    ('how much budget do we have',    'budget'),
    ('who is my co-founder',          'cofounder'),
    ('which region do we deploy to',  'region'),
    ('what is our error budget',      'errorbudget'),
    ('who is the customer',           'customer'),
    ('how should you format replies', 'style'),
]

def build_conversation():
    rng = random.Random(42)
    planted = {t: (m, k, num) for t, m, k, num in FACTS}
    turns = []
    for t in range(1, N_TURNS + 1):
        text, key, num = planted.get(t, (rng.choice(FILLER), None, False))
        # each turn costs ~410 tokens on the wire (user msg + assistant reply), cf. notebook 1
        turns.append(dict(turn=t, text=text, key=key, numeric=num,
                          tokens=int(rng.gauss(410, 90))))
    return turns

CONVERSATION = build_conversation()
TOTAL_TOKENS = sum(m['tokens'] for m in CONVERSATION)
print(f'Conversation: {N_TURNS} turns, {TOTAL_TOKENS:,} tokens.')
print(f'Context budget: {CONTEXT_BUDGET_TOKENS:,} tokens '
      f'({CONTEXT_BUDGET_TOKENS/TOTAL_TOKENS:.0%} of it fits).')
print(f'{len(FACTS)} facts are planted at turns {[t for t,_,_,_ in FACTS]}.')


In [ ]:
# ==== The four strategies ====
STOPWORDS = set('what when who which how is are do does the a an of to my our we i you me '
                'for in on and that it this be have has should'.split())
def keywords(s):
    return {w.strip('.,?').lower() for w in s.split()} - STOPWORDS

def fit_recent(turns, budget):
    """Greedily keep the most recent turns that fit in `budget` tokens."""
    kept, used = [], 0
    for m in reversed(turns):
        if used + m['tokens'] > budget:
            break
        kept.append(m); used += m['tokens']
    return list(reversed(kept)), used

def summarise(older, rng):
    """Model a summariser: keeps a fraction of facts, blurs some numbers, drops filler."""
    out = []
    for m in older:
        if m['key'] is None:                       # filler never survives a summary
            continue
        if rng.random() < SUMMARY_FACT_RETENTION:
            degraded = m['numeric'] and rng.random() < SUMMARY_PRECISION_LOSS
            out.append(('summary_degraded' if degraded else 'summary', m))
    return out

def strat_truncate(probe):
    kept, used = fit_recent(CONVERSATION, CONTEXT_BUDGET_TOKENS)
    return [('verbatim', m) for m in kept], used, 0

def strat_summarise(probe):
    recent, used = fit_recent(CONVERSATION, CONTEXT_BUDGET_TOKENS - SUMMARY_TOKENS)
    older = [m for m in CONVERSATION if m not in recent]
    items = summarise(older, random.Random(7))
    return items + [('verbatim', m) for m in recent], used + SUMMARY_TOKENS, 1

def strat_retrieve(probe):
    recent, used = fit_recent(CONVERSATION, RECENT_VERBATIM_TURNS * 450)
    archive = [m for m in CONVERSATION if m not in recent]
    q = keywords(probe)
    ranked = sorted(archive, key=lambda m: (-len(q & keywords(m['text'])), m['turn']))
    picked = []
    for m in ranked:
        if not (q & keywords(m['text'])):                 # nothing matches: stop
            break
        if used + m['tokens'] > CONTEXT_BUDGET_TOKENS:
            break
        picked.append(m); used += m['tokens']
    return [('verbatim', m) for m in picked + recent], used, 0

def strat_hybrid(probe):
    recent, used = fit_recent(CONVERSATION, RECENT_VERBATIM_TURNS * 450)
    archive = [m for m in CONVERSATION if m not in recent]
    items = summarise(archive, random.Random(7))
    used += SUMMARY_TOKENS
    q = keywords(probe)
    ranked = sorted(archive, key=lambda m: (-len(q & keywords(m['text'])), m['turn']))
    for m in ranked:
        if not (q & keywords(m['text'])) or used + m['tokens'] > CONTEXT_BUDGET_TOKENS:
            break
        items.append(('verbatim', m)); used += m['tokens']
    return items + [('verbatim', m) for m in recent], used, 1

STRATEGIES = [
    ('full history (impossible)', None),
    ('truncate to recent',        strat_truncate),
    ('rolling summary',           strat_summarise),
    ('retrieval (lexical)',       strat_retrieve),
    ('hybrid (all three)',        strat_hybrid),
]

def context_for(fn, probe):
    if fn is None:
        return [('verbatim', m) for m in CONVERSATION], TOTAL_TOKENS, 0
    return fn(probe)

print(f'{"strategy":<27} {"exact":>6} {"blurred":>8} {"LOST":>5} '
      f'{"avg ctx tokens":>15} {"extra LLM calls":>16}')
print('-' * 82)
for name, fn in STRATEGIES:
    exact = blurred = lost = tok_sum = calls = 0
    for probe, need in PROBES:
        items, used, c = context_for(fn, probe)
        found = next((kind for kind, m in items if m['key'] == need), None)
        if   found in ('verbatim', 'summary'): exact += 1
        elif found == 'summary_degraded':      blurred += 1
        else:                                  lost += 1
        tok_sum += used; calls += c
    print(f'{name:<27} {exact:>6} {blurred:>8} {lost:>5} '
          f'{tok_sum//len(PROBES):>15,} {calls/len(PROBES):>16.1f}')


Now the same strategies against a question that is **not** a lookup — "summarise everything we
agreed". This needs *global* coverage, not one relevant fact, and it flips the ranking.


In [ ]:
# ==== The query retrieval cannot answer ====
all_keys = {k for _, _, k, _ in FACTS}
print(f'{"strategy":<27} {"facts present in context":>26} {"coverage":>10}')
print('-' * 66)
for name, fn in STRATEGIES:
    items, used, _ = context_for(fn, 'summarise everything we agreed so far')
    present = {m['key'] for _, m in items if m['key']}
    print(f'{name:<27} {len(present):>18} of {len(all_keys)} {len(present)/len(all_keys):>10.0%}')

print()
print('Retrieval wins the lookup table and loses this one. Top-k retrieval is, by')
print('construction, a way of NOT looking at most of your data — so any question whose')
print('answer is spread across the whole conversation ("what have we decided?", "how many')
print('times did I change my mind?", "what did I say first?") is invisible to it.')
print('Summarisation is the only strategy here that degrades gracefully on those.')


**What each strategy actually loses:**

- **Truncation** loses the *beginning*, which is where people put the durable facts — the
  allergy, the name, the constraint that must hold for the whole conversation. It fails
  silently and confidently: the model does not know something was removed, so it does not say
  "I don't remember", it just answers as if you never mentioned it. This is the worst failure
  mode of the three because it is invisible.
- **Summarisation** loses *precision*, and disproportionately loses **numbers, names and
  negations** — exactly the tokens where being approximately right is being wrong. "About
  4,000 euros" instead of "4,200 euros" is not a rounding error when it goes into an invoice.
  It also costs an extra model call per compaction: latency on some turns, money on all of
  them, and a new failure mode (what happens when the summariser is down?).
- **Retrieval** loses whatever the query doesn't lexically or semantically match. It is
  excellent at "what did I say about X" and structurally incapable of "what have we discussed"
  or "what did I say third". A dense embedding model would rescue the `allergy` probe above
  (food ↔ peanuts) — and would introduce the false-match failure we measure in the caching
  section.
- **Hybrid** is the production answer and the most expensive: recent verbatim + rolling
  summary + retrieval over the archive. Three subsystems, three ways to be stale, and an
  eval suite you now have to maintain to know whether a change made memory better or worse.

⚖️ **The trade-off nobody puts on the slide:** every one of these makes the product
*non-deterministic across turns*. The same question asked at turn 5 and turn 35 gets different
context and therefore possibly different answers. Users experience that as the assistant
"forgetting", and no amount of retrieval quality fully removes it — you have merely chosen
which things it forgets.


## 🎟️ Metering: tokens, not requests

Every rate limiter you have written counts requests. Here that is the wrong unit, and being
wrong about it is expensive rather than merely inelegant.

Two users, both well-behaved by a request-based limit:


In [ ]:
# ==== Weighting: what does a token actually cost? ====
# Derived from notebook 2's cost model rather than invented. A prefill token costs
# 1/PREFILL_TOK_S seconds; a decode token costs one step shared across the batch.
PREFILL_TOK_S     = 20_000
WEIGHT_STREAM_S   = 0.015
PER_SEQ_S         = 0.00005
TYPICAL_BATCH     = 64

sec_per_input_token  = 1 / PREFILL_TOK_S
sec_per_output_token = (WEIGHT_STREAM_S + PER_SEQ_S * TYPICAL_BATCH) / TYPICAL_BATCH
OUTPUT_WEIGHT = sec_per_output_token / sec_per_input_token

print(f'GPU-seconds per input token : {sec_per_input_token*1e6:>8.2f} µs')
print(f'GPU-seconds per output token: {sec_per_output_token*1e6:>8.2f} µs')
print(f'=> 1 output token costs {OUTPUT_WEIGHT:.0f} input tokens. That is our billing weight.')
print()
print('Note this weight is not a constant of nature: it falls out of the batch size.')
print('At batch 8 it would be', end=' ')
_b = 8
print(f'{((WEIGHT_STREAM_S + PER_SEQ_S*_b)/_b) / sec_per_input_token:.0f}x, at batch 256 it would be', end=' ')
_b = 256
print(f'{((WEIGHT_STREAM_S + PER_SEQ_S*_b)/_b) / sec_per_input_token:.0f}x.')
print('Your prices encode how well-batched your fleet is.')
print()
print('Notebook 1 assumed a 10:1 prefill:decode throughput ratio per GPU; this replica-level')
print('model lands lower because it charges prefill for the whole replica. Both are')
print('assumptions in the same ballpark. What matters is that the weight is >1 and that you')
print('derive it from your own measured batch size rather than copying a competitor\'s price.')


In [ ]:
# ==== ❌ BAD: limit by requests ====
REQUEST_LIMIT_PER_MIN = 60

users = {
    'chatty':  dict(requests=60, in_tokens=200,     out_tokens=100),
    'whale':   dict(requests=10, in_tokens=100_000, out_tokens=2_000),
}

def weighted_tokens(u):
    return u['requests'] * (u['in_tokens'] + u['out_tokens'] * OUTPUT_WEIGHT)

print(f'{"user":<8} {"requests":>9} {"in tok":>10} {"out tok":>9} '
      f'{"request limit":>14} {"weighted tokens":>17}')
print('-' * 74)
for name, u in users.items():
    ok = 'under' if u['requests'] <= REQUEST_LIMIT_PER_MIN else 'OVER'
    print(f'{name:<8} {u["requests"]:>9} {u["in_tokens"]*u["requests"]:>10,} '
          f'{u["out_tokens"]*u["requests"]:>9,} {ok:>14} {weighted_tokens(u):>17,.0f}')

ratio = weighted_tokens(users['whale']) / weighted_tokens(users['chatty'])
print()
print(f'Both users are INSIDE a {REQUEST_LIMIT_PER_MIN} req/min limit.')
print(f'The whale consumes {ratio:.0f}x the GPU of the chatty one.')
print('A request-based limiter does not throttle abuse here; it throttles the cheap user')
print('and waves the expensive one through. Worse, the incentive it creates is exactly')
print('backwards: it teaches clients to batch more work into fewer, larger requests.')


### ✅ Token buckets — and the problem you only hit with LLMs

Switch the bucket to weighted tokens and the accounting is right. But there is a wrinkle that
does not exist in any other rate limiter you have built:

> **You cannot know the cost of the request until you have finished serving it.**

Input tokens you can count up front. Output tokens are decided by the model, one at a time,
over the next thirty seconds. So a naive "charge on completion" limiter lets a user with
1 token of quota left start a 100,000-token generation.

The fix is a **reservation**: debit `input + max_tokens × weight` at admission, refund the
difference when the generation ends. That is correct, and it costs you something real.


In [ ]:
class TokenBucket:
    """Weighted token bucket with reserve-then-refund, so cost is charged BEFORE it is spent."""
    def __init__(self, capacity, refill_per_sec):
        self.capacity = capacity
        self.tokens = float(capacity)
        self.refill = refill_per_sec
        self.last = 0.0
    def _advance(self, now):
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.refill)
        self.last = now
    def reserve(self, now, cost):
        self._advance(now)
        if self.tokens < cost:
            return False
        self.tokens -= cost
        return True
    def refund(self, amount):
        self.tokens = min(self.capacity, self.tokens + amount)

QUOTA          = 200_000        # weighted tokens
REFILL_PER_SEC = QUOTA / 3600   # a full bucket per hour
ADVERTISED_MAX_TOKENS = 4_096

def run_policy(policy, requests):
    """requests: list of (input_tokens, actual_output_tokens)"""
    b = TokenBucket(QUOTA, REFILL_PER_SEC)
    admitted = overshoot = 0
    for t, (inp, out) in enumerate(requests):
        actual = inp + out * OUTPUT_WEIGHT
        if policy == 'charge_after':                       # naive
            if b.reserve(t, 0):                            # admits unconditionally
                admitted += 1
                before = b.tokens
                b.tokens -= actual
                if b.tokens < 0:                           # went negative: quota blown
                    overshoot += -b.tokens
                    b.tokens = 0.0
        elif policy == 'reserve_max':                      # correct, conservative
            hold = inp + ADVERTISED_MAX_TOKENS * OUTPUT_WEIGHT
            if b.reserve(t, hold):
                admitted += 1
                b.refund(hold - actual)
    return admitted, overshoot

# 40 modest requests, then one enormous one at the end.
normal = [(800, 300)] * 40
abusive = normal + [(120_000, 4_000)]

print(f'{"policy":<14} {"workload":<26} {"admitted":>9} {"quota overshoot":>17}')
print('-' * 70)
for pol in ('charge_after', 'reserve_max'):
    for label, wl in (('40 normal requests', normal),
                      ('40 normal + 1 huge', abusive)):
        a, o = run_policy(pol, wl)
        print(f'{pol:<14} {label:<26} {a:>9} {o:>17,.0f}')

print()
print('`charge_after` cannot stop the last request, because it does not know what it will')
print('cost until it has already paid for it. The quota is a suggestion.')
print('`reserve_max` holds the worst case up front, so the huge request is rejected before')
print('a single GPU-second is spent on it.')


In [ ]:
# ==== What reserve-then-refund costs an honest user ====
# The reservation is based on max_tokens, but real replies are short. A user who never
# comes close to max_tokens still has their effective quota cut.
typical_actual = 800 + 300 * OUTPUT_WEIGHT
print(f'A typical request really costs {typical_actual:,.0f} weighted tokens, so an honest user')
print(f'should get {QUOTA/typical_actual:.0f} of them out of a {QUOTA:,}-token quota. What they actually get:')
print()
print(f'{"max_tokens":>11} | {"reservation held":>17} | {"requests before empty":>22} | quota lost')
print('-' * 74)
for mt in (512, 1_024, 4_096, 16_384):
    held = 800 + mt * OUTPUT_WEIGHT
    n_req = QUOTA / held
    print(f'{mt:>11,} | {held:>17,.0f} | {n_req:>22.0f} | '
          f'{1 - n_req/(QUOTA/typical_actual):>9.0%}')
print()
print('(Dropping max_tokens below the typical reply length would "fix" the quota by')
print(' truncating answers instead — the same bad trade the scheduler offered in')
print(' notebook 2, wearing different clothes.)')

print()
print('⚖️  The honest trade-offs of reserve-then-refund:')
print('  - It shrinks effective quota for well-behaved users, in proportion to how generous')
print('    your max_tokens default is. Tighten the default and you fix the quota but')
print('    truncate long answers (notebook 2 showed the same tension in the scheduler).')
print('  - Concurrency gets limited as a side effect: N in-flight requests hold N')
print('    reservations, so a user can be refused not for using too much but for having')
print('    too many open at once. Sometimes that is what you want; say so deliberately.')
print('  - Refunds must be idempotent and crash-safe. A generation that dies without')
print('    refunding leaks quota, and the user is throttled for an hour by your bug.')
print('    Give every reservation a TTL and reclaim it on expiry.')
print('  - Buckets live in Redis and are shared across API nodes, so the check is a')
print('    network round trip on the hot path. Batch it, or accept a small per-node')
print('    overshoot and reconcile centrally. Perfect global accuracy is not worth 5ms.')


## 🔁 Idempotency and retries for a partially-delivered stream

Here is the hardest reliability problem in the design, and the one with no clean answer.

A normal API retry is safe because the response is atomic: either the client got it or it
didn't. A streamed generation is **neither**. The user has already read 120 tokens when the
connection dies. Now what?

Everything that normally makes retries safe is unavailable:

- **You cannot re-send a status code.** `200 OK` went out with the first token. You cannot
  turn the response into a `503` afterwards.
- **You cannot reproduce the answer.** Sampling at temperature > 0 means running the same
  prompt again produces *different text*. There is no "just retry" that yields the same
  bytes — which means the only way to resume is to have **stored what you already sent**.
- **You cannot tell "the client left" from "the network broke".** Both look like a closed
  socket, and they want opposite handling (cancel vs preserve).
- **Billing already happened**, at least partially. The GPU-seconds are spent whether or not
  the bytes arrived.


In [ ]:
import zlib

VOCAB = ('the model streams tokens one at a time which makes retries interesting '
         'because part of the answer has already reached the user').split()

class GenerationStore:
    """Durable record of a generation, keyed by a client-supplied idempotency key.

    The token log is the whole trick: because the model cannot reproduce its own output,
    the ONLY source of truth for 'what did we already send' is what we wrote down.
    """
    def __init__(self):
        self.by_key = {}      # idempotency_key -> generation_id
        self.records = {}     # generation_id  -> record
        self.gpu_tokens_spent = 0

    def start_or_attach(self, key, total_tokens):
        if key in self.by_key:                       # replay: never generate twice
            return self.records[self.by_key[key]], False
        # A fresh generation id per ATTEMPT, and a fresh sample seed with it: this is the
        # point -- the model does not reproduce its own output, so attempt 2 says something
        # different from attempt 1 even for a byte-identical prompt.
        gen_id = f'gen-{len(self.records):04d}'
        rng = random.Random(zlib.crc32(gen_id.encode()))
        rec = dict(gen_id=gen_id, key=key, status='running',
                   tokens=[rng.choice(VOCAB) for _ in range(total_tokens)],
                   emitted=0)
        self.by_key[key] = gen_id
        self.records[gen_id] = rec
        self.gpu_tokens_spent += total_tokens
        return rec, True

    def read_from(self, rec, seq):
        """Serve the token log from sequence `seq` onwards (SSE Last-Event-ID)."""
        return rec['tokens'][seq:]

def deliver(rec, store, start_seq, drop_at=None):
    """Stream tokens to a client. Returns (tokens the client received, next seq)."""
    got = []
    for seq, tok in enumerate(store.read_from(rec, start_seq), start=start_seq):
        if drop_at is not None and seq >= drop_at:
            return got, seq                              # connection died here
        got.append(tok)
    rec['status'] = 'complete'
    return got, len(rec['tokens'])


In [ ]:
# ==== Three client behaviours, same failure ====
TOTAL = 300
DROP_AT = 120

def scenario(label, use_idempotency_key, use_resume):
    store = GenerationStore()
    rendered = []                                        # what the USER ends up seeing
    key = 'msg-abc-123' if use_idempotency_key else 'attempt-1'
    rec, _ = store.start_or_attach(key, TOTAL)
    got, next_seq = deliver(rec, store, 0, drop_at=DROP_AT)
    rendered += got

    # --- connection dropped; the client retries ---
    retry_key = key if use_idempotency_key else 'attempt-2'
    rec2, fresh = store.start_or_attach(retry_key, TOTAL)
    resume_from = next_seq if use_resume else 0
    got2, _ = deliver(rec2, store, resume_from)
    rendered += got2

    duplicated = max(0, len(rendered) - TOTAL)
    answers = len({rec['gen_id'], rec2['gen_id']})
    same_text = rec['tokens'][:DROP_AT] == rec2['tokens'][:DROP_AT]
    print(f'{label:<34} rendered={len(rendered):>4} | re-sent={duplicated:>4} | '
          f'billed={store.gpu_tokens_spent:>4} | generations={answers} | '
          f'2nd attempt matches 1st: {same_text}')
    return rendered

print(f'A {TOTAL}-token reply, connection drops after {DROP_AT} tokens, client retries.\n')
a = scenario('❌ no key, plain retry',        False, False)
b = scenario('🙂 idempotency key, no resume', True,  False)
c = scenario('✅ idempotency key + resume',   True,  True)

print()
print(f'❌  Two generations ran, so the user was billed for {2*TOTAL} tokens AND the second')
print('    attempt produced different text — see the last column. The user reads 120')
print('    tokens of one answer followed by 300 of another. The UI must choose which')
print('    half-answer to keep, and the conversation history may end up with both.')
print(f'🙂  One generation, correct billing — but all {TOTAL} tokens are re-sent, so the')
print(f'    client renders {DROP_AT} of them twice unless it knows to discard the prefix.')
print('    Acceptable, and much better than nothing: the answer is at least self-consistent.')
print('✅  One generation, correct billing, zero duplicated tokens. The client sends')
print('    Last-Event-ID and the server serves the tail of the stored token log.')


### The genuinely hard case: the *server* died

Everything above assumed the record survived. Now suppose the replica holding the generation
crashed at token 120. The KV cache is gone. The token log has 120 tokens and status
`running` with a lease that has expired.

There is no good option, only a choice between three bad ones:

| Option | What the user sees | Cost |
|---|---|---|
| **Fail the turn** | error, "please try again" | honest, but you throw away 120 tokens of work the user already read |
| **Regenerate from scratch** | the answer *changes under them* mid-read | full re-prefill + full decode, and a jarring UX |
| **Continue from the prefix** | one coherent answer | re-prefill of `prompt + 120 tokens`, and the continuation may not match the earlier style |

**Continue-from-prefix is usually right**, and it is only possible because you wrote the token
log down. Let's price it.


In [ ]:
PROMPT_TOKENS = 1_500

def resume_cost(generated_so_far, prompt_tokens=PROMPT_TOKENS, total=TOTAL):
    restart  = dict(prefill=prompt_tokens,                    decode=total)
    continue_ = dict(prefill=prompt_tokens + generated_so_far, decode=total - generated_so_far)
    return restart, continue_

print(f'{"crashed at":>11} | {"restart: prefill+decode":>25} | '
      f'{"continue: prefill+decode":>26} | continue saves')
print('-' * 88)
for at in (10, 120, 250, 295):
    r, c = resume_cost(at)
    # weight by notebook 2's relative cost: a decode token is OUTPUT_WEIGHT prefill tokens
    r_cost = r['prefill'] + r['decode'] * OUTPUT_WEIGHT
    c_cost = c['prefill'] + c['decode'] * OUTPUT_WEIGHT
    print(f'{at:>11} | {r["prefill"]:>10,} + {r["decode"]:>10,} | '
          f'{c["prefill"]:>11,} + {c["decode"]:>11,} | {1 - c_cost/r_cost:>13.0%}')

print()
print('Continuing is cheaper the later the crash happens, which is exactly when it hurts')
print('the user most — so the incentives line up for once.')
print()
print('⚖️  What continue-from-prefix costs you, honestly:')
print('  - The continuation is a different sample. Tone, formatting and even the position')
print('    it thinks it is in can shift mid-answer. It reads like two people finishing one')
print('    sentence. Users notice.')
print('  - You must persist the token log durably and cheaply, on the hot path, for every')
print('    generation — most of which will never need it. Buffer in Redis with a short TTL,')
print('    flush the final message to the durable store, and accept losing the tail of a')
print('    generation whose replica AND cache both die.')
print('  - Leases and fencing. Two workers must never resume the same generation. That')
print('    means a lease with a token that the store checks on every append.')
print('  - "Partially delivered" becomes a first-class state in your data model, your')
print('    billing, your moderation pipeline and your analytics. There is no way to keep')
print('    it contained to the streaming layer.')


### The rules, stated plainly

1. **The client generates the idempotency key**, not the server, and it is stable across
   retries of the same user action. A UUIDv7 per user message is the natural choice.
2. **The server's first act is to write the record**, before any GPU work. Concurrent retries
   then attach to one generation rather than starting two.
3. **Every token has a sequence number** in the log and on the wire.
4. **Resume beats restart; restart beats failing; failing beats duplicating.**
5. **Bill the generation, not the delivery.** Tokens produced cost money whether or not they
   reached the client — but bill *once per idempotency key*, never once per attempt.
6. **Persist incrementally with a `status`.** A crash should leave a truncated message marked
   incomplete, never a hole in the conversation.


## 💾 Caching

Three different caches, doing three different jobs. Only one of them is safe by default.


In [ ]:
# ==== 1. Exact-match response cache ====
# Key: hash of (model, params, the ENTIRE message list). Safe -- identical input, identical
# semantics. The problem is that conversation context makes identical input rare.
import hashlib

def cache_key(model, temperature, messages):
    h = hashlib.sha256()
    h.update(f'{model}|{temperature}|'.encode())
    for role, content in messages:
        h.update(f'{role}:{content}\x00'.encode())
    return h.hexdigest()[:16]

# ASSUMPTIONS about how repetitive first messages are. This is the whole hit rate.
HEAD_SHARE     = 0.25    # share of first messages drawn from a small popular set
HEAD_QUESTIONS = 500     # size of that popular set

rng = random.Random(5)
print(f'{"turn depth":>11} | {"exact-match hit rate":>21}')
print('-' * 37)
for depth in (1, 2, 3, 5):
    seen, hits, N = set(), 0, 20_000
    for _ in range(N):
        if rng.random() < HEAD_SHARE:
            first = f'faq-{rng.randrange(HEAD_QUESTIONS)}'      # a popular question
        else:
            first = f'unique-{rng.randrange(10**9)}'            # long tail
        msgs = [('user', first)]
        for _d in range(1, depth):                              # later turns are unique
            msgs.append(('assistant', f'reply-{rng.randrange(10**9)}'))
            msgs.append(('user', f'followup-{rng.randrange(10**9)}'))
        k = cache_key('gpt-x', 0.0, msgs)
        if k in seen: hits += 1
        seen.add(k)
    print(f'{depth:>11} | {hits/N:>20.1%}')

print()
print(f'Turn 1 hits at roughly HEAD_SHARE ({HEAD_SHARE:.0%}) once the popular set is warm.')
print('From turn 2 onward it is ~0%, because the key contains a conversation history that')
print('has never existed before and never will again.')
print()
print('Two more limits worth stating out loud:')
print('  - It is only valid at temperature 0. Above that, two identical requests are')
print('    SUPPOSED to give different answers, and caching quietly removes that.')
print('  - The key must include everything that changes the answer: model version, system')
print('    prompt, tools, retrieved documents, user locale. Miss one and you serve a')
print('    stale answer forever.')
print('Worth having. Not where the money is.')


In [ ]:
# ==== 2. Prefix / KV caching -- the one that actually pays ====
# Turn t's prompt is turn (t-1)'s prompt plus the last reply plus the new message. The
# shared prefix's KV can be reused instead of recomputed. Numbers from notebook 1.
S, u, r, T = 400, 60, 350, 6

no_cache = sum(S + (t - 1) * (u + r) + u for t in range(1, T + 1))
with_cache = (S + u) + sum(r + u for _ in range(T - 1))   # only NEW tokens need prefill

print(f'{"turn":>5} | {"prompt":>8} | {"prefill without cache":>22} | {"with prefix cache":>18}')
print('-' * 62)
for t in range(1, T + 1):
    prompt = S + (t - 1) * (u + r) + u
    new = (S + u) if t == 1 else (r + u)
    print(f'{t:>5} | {prompt:>8,} | {prompt:>22,} | {new:>18,}')
print('-' * 62)
print(f'{"TOTAL":>5} | {"":>8} | {no_cache:>22,} | {with_cache:>18,}')
print()
print(f'Prefix caching removes {1 - with_cache/no_cache:.0%} of all prefill work in this conversation.')
print('Notebook 1 sized 1,779 GPUs for prefill; this would take it to roughly '
      f'{1779*with_cache/no_cache:,.0f}.')
print()
print('⚖️  What it costs: the cached prefix occupies KV memory that could have held an')
print('    active generation, so it trades notebook 2\'s scarcest resource for compute.')
print('    It also makes routing sticky — a request should go to the replica that already')
print('    holds its prefix, which fights against load balancing. And the system prompt')
print('    must be byte-identical: change one character and every prefix in the fleet')
print('    misses at once. That is a deploy-shaped cliff, not a gradual degradation.')


### 3. Semantic caching — and when it lies

The idea: embed the query, find the nearest cached question, and if it is close enough, return
that cached answer. Hit rates go up enormously because paraphrases now hit.

The problem is that "close in embedding space" and "means the same thing" are **not the same
relation**, and they come apart in a specific, predictable way: embeddings encode *topic*
strongly and *polarity* weakly. "Enable 2FA" and "disable 2FA" are about the same thing.

Below we build a toy embedding with exactly that property — words in the same topic get a
shared component — and measure what it does. **This is a deliberately constructed
demonstration of a known failure mode, not a measurement of any real embedding model.** But
the failure it demonstrates is real, and it is why semantic caching is dangerous by default.


In [ ]:
import math, zlib

DIM = 64
TOPIC_WEIGHT = 0.80     # how much of a word's meaning is "topic" vs "the specific word"
TOPICS = {
    'enable': '2fa_toggle', 'disable': '2fa_toggle', 'turn': '2fa_toggle',
    'on': '2fa_toggle', 'off': '2fa_toggle',
    '2fa': '2fa', 'two-factor': '2fa', 'mfa': '2fa',
    'reset': 'pwreset', 'change': 'pwreset', 'password': 'pwreset',
    'paris': 'city', 'berlin': 'city',
    'weather': 'weather', 'forecast': 'weather',
    'cancel': 'sub_action', 'renew': 'sub_action', 'subscription': 'sub',
    'safe': 'safety', 'ibuprofen': 'drug', 'alcohol': 'drug',
    'with': 'combine', 'without': 'combine',
}
STOP_EMB = set('how do i what is the a an way in my of to can you'.split())

def _unit(seed):
    rng = random.Random(seed)
    v = [rng.gauss(0, 1) for _ in range(DIM)]
    n = math.sqrt(sum(x * x for x in v))
    return [x / n for x in v]

def word_vec(w):
    base = _unit(zlib.crc32(('w:' + w).encode()))       # crc32, not hash(): deterministic
    topic = TOPICS.get(w)
    if topic is None:
        return base
    tv = _unit(zlib.crc32(('t:' + topic).encode()))
    mix = [TOPIC_WEIGHT * a + (1 - TOPIC_WEIGHT) * b for a, b in zip(tv, base)]
    n = math.sqrt(sum(x * x for x in mix))
    return [x / n for x in mix]

def embed(s):
    ws = [w.strip('?.,').lower() for w in s.split()]
    ws = [w for w in ws if w not in STOP_EMB] or ws
    vs = [word_vec(w) for w in ws]
    m = [sum(c) / len(vs) for c in zip(*vs)]
    n = math.sqrt(sum(x * x for x in m)) or 1.0
    return [x / n for x in m]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

# (incoming query, cached question, is serving the cached answer CORRECT?)
PAIRS = [
    ('how do I reset my password',      'what is the way to reset a password', True),
    ('how do I change my password',     'what is the way to reset a password', True),
    ('how do I turn on two-factor',     'how do I enable 2fa',                 True),
    ('what is the forecast in paris',   'what is the weather in paris',        True),
    ('how do I enable 2fa',             'how do I disable 2fa',                False),
    ('how do I turn off two-factor',    'how do I turn on two-factor',         False),
    ('what is the weather in paris',    'what is the weather in berlin',       False),
    ('is ibuprofen safe with alcohol',  'is ibuprofen safe without alcohol',   False),
    ('how do I cancel my subscription', 'how do I renew my subscription',      False),
    ('how do I reset my password',      'what is the weather in berlin',       False),
]

print(f'{"incoming query":<33} {"nearest cached question":<37} {"cos":>6}  serving it would be')
print('-' * 100)
for q, c, ok in sorted(PAIRS, key=lambda p: -cosine(embed(p[0]), embed(p[1]))):
    print(f'{q:<33} {c:<37} {cosine(embed(q), embed(c)):>6.3f}  '
          f'{"correct" if ok else "A WRONG ANSWER"}')
print()
print('Sort that column and the problem is immediate: the answers that would be WRONG are')
print('interleaved with -- and often score higher than -- the ones that would be right.')


In [ ]:
# ==== Is there a safe threshold? ====
print(f'{"threshold":>10} | {"correct hits":>13} | {"WRONG answers":>14} | '
      f'{"missed":>7} | precision')
print('-' * 66)
for th in (0.995, 0.99, 0.98, 0.97, 0.95, 0.90, 0.50):
    tp = sum(1 for q, c, ok in PAIRS if ok and cosine(embed(q), embed(c)) >= th)
    fp = sum(1 for q, c, ok in PAIRS if not ok and cosine(embed(q), embed(c)) >= th)
    fn = sum(1 for q, c, ok in PAIRS if ok and cosine(embed(q), embed(c)) < th)
    prec = tp / (tp + fp) if tp + fp else float('nan')
    print(f'{th:>10.3f} | {tp:>13} | {fp:>14} | {fn:>7} | {prec:>9.2f}')

print()
print('There is no threshold that gets all the paraphrases and none of the polarity flips,')
print('because the wrong answers score HIGHER than some of the right ones. Tightening the')
print('threshold does not make the cache safe; it just makes it a worse exact-match cache.')


⚖️ **When semantic caching returns a wrong answer — and what to do about it**

The failures are not random. They cluster:

- **Negation and polarity.** *enable/disable*, *with/without*, *cancel/renew*, *should/should
  not*. The single most dangerous class, because the wrong answer is confident, fluent, and
  the exact opposite of correct.
- **Entity swaps.** *Paris/Berlin*, *2023/2024*, *Alice's account/Bob's account*. Same
  question shape, different subject.
- **Anything personal or stateful.** "What's my balance", "when is my flight". These are
  identical *as text* between users. No similarity threshold saves you, because the
  similarity is genuinely 1.0 — the query text simply is not the whole cache key.
- **Anything time-sensitive.** A correct cached answer becomes a wrong one at midnight.

So the rule is not "tune the threshold". It is **bound the blast radius**:

1. Namespace the cache by everything that changes the answer — user or tenant id, tools
   available, model, system prompt version, retrieved documents, and a date bucket.
2. Enable it only for a whitelisted class of queries (public FAQ-shaped) and never for
   personalised, transactional, medical, legal or financial ones.
3. Treat a semantic hit as a *candidate*, and have a cheap verifier confirm it — which costs
   a small model call and eats much of the saving. That is the real trade: semantic caching
   is only cheap while it is unsafe.
4. Log hits with their similarity and sample them for review. A semantic cache that nobody
   audits is an unmonitored source of confidently wrong answers.

Honestly: **prefix/KV caching is where the money is, and it is always correct.** Semantic
caching is a much smaller win for a much larger risk. Reach for it last.


## 🛡️ Safety and moderation as a pipeline stage

Moderation is two checks, and the second one is made genuinely hard by streaming.

```
   user input ──▶ [input moderation] ──▶ [generate] ──▶ [output moderation] ──▶ user
                        │                                      │
                   blocks before                       must decide about text
                   any GPU is spent                    that is ALREADY LEAVING
```

Input moderation is easy: a small classifier, a few tens of milliseconds, and it runs before
you spend a GPU-second — so it *saves* money on the requests it blocks.

Output moderation is the problem. You are streaming tokens as they are produced. To check a
sentence you need the sentence. To have the sentence, you must have already generated it — and
if you have already *sent* it, checking is pointless.


In [ ]:
# ==== ASSUMPTIONS: moderation ====
INPUT_MOD_MS         = 30      # small classifier on the prompt    [range: 5 - 300]
OUTPUT_MOD_MS        = 20      # per chunk checked                 [range: 5 - 200]
UNSAFE_RATE          = 0.02    # share of generations containing an unsafe span
UNSAFE_SPAN_TOKENS   = 25      # how long such a span is
TOKENS_PER_SEC       = 50      # streaming speed from notebook 2
N_GENERATIONS        = 5_000
GEN_TOKENS           = 300

def moderation_policy(policy, hold_back=0):
    """Returns (added TTFT ms, unsafe tokens shown to users, retractions)."""
    rng = random.Random(99)
    leaked = retracted = 0
    for _ in range(N_GENERATIONS):
        has_unsafe = rng.random() < UNSAFE_RATE
        if policy == 'stream_raw':
            if has_unsafe: leaked += UNSAFE_SPAN_TOKENS
            ttft_ms = INPUT_MOD_MS
        elif policy == 'moderate_at_end':
            ttft_ms = INPUT_MOD_MS + GEN_TOKENS / TOKENS_PER_SEC * 1000 + OUTPUT_MOD_MS
        elif policy == 'buffered_window':
            # hold back `hold_back` tokens; a span is caught if it completes inside the buffer
            ttft_ms = INPUT_MOD_MS + hold_back / TOKENS_PER_SEC * 1000 + OUTPUT_MOD_MS
            if has_unsafe and UNSAFE_SPAN_TOKENS > hold_back:
                leaked += UNSAFE_SPAN_TOKENS - hold_back
                retracted += 1
    return ttft_ms, leaked, retracted

print(f'{"policy":<28} {"added TTFT":>12} {"unsafe tokens shown":>21} {"retractions":>12}')
print('-' * 78)
for label, pol, hb in (
        ('stream raw (no output mod)', 'stream_raw', 0),
        ('moderate after full gen',    'moderate_at_end', 0),
        ('buffered window, 10 tok',    'buffered_window', 10),
        ('buffered window, 25 tok',    'buffered_window', 25),
        ('buffered window, 60 tok',    'buffered_window', 60)):
    ttft, leaked, retr = moderation_policy(pol, hb)
    print(f'{label:<28} {ttft:>10.0f}ms {leaked:>21,} {retr:>12,}')

print()
print('The shape of the trade is clear: safety costs TTFT, linearly in how much text you')
print('hold back before showing it. Moderating only at the end is perfectly safe and')
print(f'destroys the product — TTFT becomes {GEN_TOKENS/TOKENS_PER_SEC:.0f} seconds, which is the entire reason')
print('streaming exists.')
print()
print('The buffered window is the production answer, and note WHY 25 tokens is the knee:')
print(f'it is exactly UNSAFE_SPAN_TOKENS. Buffer at least one "unit of meaning" — in')
print('practice a sentence or a code block — and you catch spans that complete inside it')
print('at a cost of a few hundred milliseconds.')


⚖️ **What moderation costs, beyond latency:**

- **You cannot unsay it.** Any span longer than the buffer is already on the user's screen.
  The client must support *retraction* — deleting rendered text — and that is a jarring
  experience you have to design deliberately, not a footnote.
- **The classifier is on the critical path of every token.** At 5,000 generations/second with
  a 25-token window you are running a classifier tens of thousands of times a second. That is
  its own fleet, its own scaling problem, and its own outage. Decide now whether moderation
  failing open or failing closed is worse for you, because you will find out either way.
- **False positives are invisible to you and infuriating to users.** A blocked answer looks
  like a broken product. Measure the false-positive rate as carefully as the miss rate; only
  one of them shows up in your incident channel.
- **Moderation and caching interact badly.** A cached answer skips generation — does it also
  skip output moderation? If yes, one poisoned cache entry is served forever. Moderate on
  write to the cache, and re-check on read when policy changes.
- **Input moderation is the cheap win**: it runs before the GPU, so blocking there costs
  ~30 ms and *saves* several GPU-seconds. Spend your latency budget there first.


## ✅ Summary

- **Storage is boring on purpose**: one partition per conversation, a denormalised sidebar
  index, `token_count` cached on every row, and an explicit `incomplete` status because
  generations die halfway.
- **Context management is lossy, always.** Truncation loses the oldest facts silently,
  summarisation loses precision (especially numbers), retrieval loses whatever the query
  doesn't match — and it cannot answer questions about the conversation as a whole. Hybrid
  wins on recall and costs three subsystems.
- **Meter in weighted tokens, not requests.** A request-based limit throttles your cheapest
  users and waves through the ones costing more than 20× as much. Reserve `max_tokens` up front and
  refund, because you cannot know the cost until you have already incurred it.
- **A partially-delivered stream cannot be retried, only resumed.** The model will not
  reproduce its own output, so the token log is the only source of truth. Client-generated
  idempotency key, sequence-numbered frames, resume-from-`Last-Event-ID`, and
  continue-from-prefix when a replica dies.
- **Prefix/KV caching removes ~70% of prefill work and is always correct. Semantic caching
  is a much smaller win with a real chance of returning the opposite of the right answer**,
  because embeddings encode topic strongly and polarity weakly. There is no safe threshold —
  only a bounded blast radius.
- **Output moderation trades TTFT for safety, linearly.** Buffer one unit of meaning; anything
  longer than the buffer is already on the screen and must be retracted.

Back to [Notebook 1 — Requirements & Architecture](./01_requirements_and_architecture.ipynb)
or [Notebook 2 — Serving & Scaling](./02_serving_and_scaling.ipynb).
